# Task042 semantic rule discovery

Train-only semantic rule workflow for task042. Replaces the visible-trained 13x13 patch bank with a geometric rule: two same-sized color-3 blocks on one diagonal imply color-8 blocks at the other diagonal corners after k-cell bbox expansion and clipping.

In [ ]:
import json
from pathlib import Path
from collections import Counter, deque
import numpy as np

TASK_ID = 'task042'
MODEL_VERSION = 'task042-semantic-diagonal-block-completion-rule'
TASK_JSON = next((p for p in [
    Path('Co_Kaggle/g3/competition_material/taskfiles/task042.json'),
    Path('competition_material/taskfiles/task042.json'),
    Path('/mnt/data/task042.json'),
] if p.exists()), None)
assert TASK_JSON is not None, 'Missing task042.json'
task = json.load(open(TASK_JSON, encoding='utf-8'))
print('MODEL_VERSION:', MODEL_VERSION)
print('TASK_JSON:', TASK_JSON)
print('examples:', {k: len(task.get(k, [])) for k in ['train', 'test', 'arc-gen']})
print('shape modes:', Counter(f"{len(ex['input'])}x{len(ex['input'][0])}" for split in ['train','test','arc-gen'] for ex in task.get(split, [])).most_common())

In [ ]:
def color_points(grid, value):
    return {(r, c) for r, row in enumerate(grid) for c, x in enumerate(row) if x == value}

def bbox(points):
    rs = [r for r, c in points]
    cs = [c for r, c in points]
    return min(rs), min(cs), max(rs), max(cs)

def comps8(points):
    pts = set(points)
    out = []
    while pts:
        start = pts.pop()
        comp = {start}
        q = deque([start])
        while q:
            r, c = q.popleft()
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):
                    if dr == 0 and dc == 0:
                        continue
                    nb = (r + dr, c + dc)
                    if nb in pts:
                        pts.remove(nb)
                        comp.add(nb)
                        q.append(nb)
        out.append(comp)
    return out

def predict_task042_semantic(grid):
    arr = np.array(grid, dtype=int)
    out = arr.copy()
    H, W = arr.shape
    for comp in comps8(color_points(grid, 3)):
        r0, c0, r1, c1 = bbox(comp)
        h = r1 - r0 + 1
        w = c1 - c0 + 1
        if h != w or h % 2 != 0:
            continue
        k = h // 2

        def block(rr, cc):
            return {(r, c) for r in range(rr, rr + k) for c in range(cc, cc + k)}

        tl = block(r0, c0)
        tr = block(r0, c0 + k)
        bl = block(r0 + k, c0)
        br = block(r0 + k, c0 + k)
        main_diag_score = len(comp & tl) + len(comp & br)
        anti_diag_score = len(comp & tr) + len(comp & bl)

        if main_diag_score >= anti_diag_score:
            target_blocks = [(r0 - k, c1 + 1), (r1 + 1, c0 - k)]  # add expanded TR + BL
        else:
            target_blocks = [(r0 - k, c0 - k), (r1 + 1, c1 + 1)]  # add expanded TL + BR

        for rr, cc in target_blocks:
            for r in range(rr, rr + k):
                for c in range(cc, cc + k):
                    if 0 <= r < H and 0 <= c < W and out[r, c] == 0:
                        out[r, c] = 8
    return out.tolist()

def eval_grid(fn, splits=('train', 'test', 'arc-gen')):
    rows = []
    right = total = 0
    first_wrong = None
    for split in splits:
        sr = st = 0
        for i, ex in enumerate(task.get(split, [])):
            pred = fn(ex['input'])
            ok = pred == ex['output']
            sr += int(ok); st += 1; right += int(ok); total += 1
            if not ok and first_wrong is None:
                first_wrong = {'split': split, 'index': i}
        rows.append((split, sr, st, sr / st if st else None))
    return {'right': right, 'total': total, 'accuracy': right / total if total else None, 'first_wrong': first_wrong, 'rows': rows}

semantic_summary = eval_grid(predict_task042_semantic)
print('semantic rule accuracy:', semantic_summary['right'], '/', semantic_summary['total'], semantic_summary['accuracy'], 'first_wrong=', semantic_summary['first_wrong'])
print('rows:', semantic_summary['rows'])
assert semantic_summary['right'] == semantic_summary['total']

In [ ]:
# Diagnostic: train-only 13x13 patch detector. It fits train but does not generalize.
RAD = 6

def patch_feat_03(grid, r, c, rad=RAD):
    a = np.asarray(grid)
    h, w = a.shape
    f = np.zeros((2, 2 * rad + 1, 2 * rad + 1), np.int8)
    for i, dr in enumerate(range(-rad, rad + 1)):
        for j, dc in enumerate(range(-rad, rad + 1)):
            rr, cc = r + dr, c + dc
            if 0 <= rr < h and 0 <= cc < w:
                col = int(a[rr, cc])
                if col == 0: f[0, i, j] = 1
                elif col == 3: f[1, i, j] = 1
    return tuple(f.flatten().tolist())

mapping = {}
conflicts = []
for ei, ex in enumerate(task['train']):
    target = (np.asarray(ex['output']) == 8).astype(np.int8)
    for r in range(10):
        for c in range(10):
            key = patch_feat_03(ex['input'], r, c)
            y = int(target[r, c])
            if key in mapping and mapping[key] != y:
                conflicts.append((ei, r, c, mapping[key], y))
            else:
                mapping[key] = y

def pred_patch_train_only(grid):
    out = np.asarray(grid).copy()
    for r in range(10):
        for c in range(10):
            if mapping.get(patch_feat_03(grid, r, c), 0):
                out[r, c] = 8
    return out.tolist()

patch_summary = eval_grid(pred_patch_train_only)
print('13x13 train-only patch-detector accuracy:', patch_summary['right'], '/', patch_summary['total'], patch_summary['accuracy'], 'first_wrong=', patch_summary['first_wrong'])
print('rows:', patch_summary['rows'])
print('unique train patch patterns:', len(mapping))
print('positive train patch-detectors:', sum(mapping.values()))
print('conflicts:', len(conflicts))

In [ ]:
final_row = {
    'task_id': TASK_ID,
    'model_version': MODEL_VERSION,
    'semantic_visible_right': semantic_summary['right'],
    'semantic_visible_total': semantic_summary['total'],
    'semantic_visible_accuracy': semantic_summary['accuracy'],
    'patch_train_only_right': patch_summary['right'],
    'patch_train_only_total': patch_summary['total'],
    'patch_train_only_accuracy': patch_summary['accuracy'],
    'status': 'semantic_rule_validated_no_onnx_export_yet',
}
print(final_row)